<a href="https://colab.research.google.com/github/DulaniDeSilva/Natural-Language-Processing-NLP-/blob/main/CLI_NanoChat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Demo/CS4182_NLP

/content/drive/MyDrive/Demo/CS4182_NLP


In [ ]:
!pwd

/content/drive/MyDrive/Demo/CS4182_NLP


Install uv

In [ ]:
%%bash
curl -LsSf https://astral.sh/uv/install.sh | sh
echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc

installing to /usr/local/bin
  uv
  uvx
everything's installed!


downloading uv 0.11.19 x86_64-unknown-linux-gnu


Clone nanochat

In [ ]:
%%bash
git clone https://github.com/karpathy/nanochat.git

Cloning into 'nanochat'...


Setup virtual environemnt

In [ ]:
import os
os.environ["PATH"] = os.environ["HOME"] + "/.local/bin:" + os.environ["PATH"]

!cd nanochat && uv venv
!cd nanochat && uv sync --extra gpu

Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 130 packages in 7ms
Prepared 81 packages in 1m 23s
░░░░░░░░░░░░░░░░░░░░ [0/81] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 81 packages in 9m 18s
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.12.15
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + anyio==4.10.0
 + async-timeout==5.0.1
 + attrs==25.3.0
 + certifi==2025.8.3
 + charset-normalizer==3.4.3
 + click==8.2.1
 + datasets==4.0.0
 + dill==0.3.8
 + exceptiongroup==1.3.0
 + fastapi==0.117.1
 + filelock==3.19.1
 + frozenlist==1.7.0
 + fsspec==2025.3.0
 + gitd

Set cache dir and train the tokenizer

In [ ]:
%%bash
export NANOCHAT_BASE_DIR="$HOME/.cache/nanochat"
mkdir -p $NANOCHAT_BASE_DIR
cd nanochat
source .venv/bin/activate

# Download ~8 data shards
python -m nanochat.dataset -n 8

# Train tokenizer on 2B characters
python -m scripts.tok_train --max-chars=2000000000

# Evaluate tokenizer
python -m scripts.tok_eval

Target directory: /root/.cache/nanochat/base_data_climbmix

Skipping /root/.cache/nanochat/base_data_climbmix/shard_00000.parquet (already exists)
Skipping /root/.cache/nanochat/base_data_climbmix/shard_00001.parquet (already exists)
Skipping /root/.cache/nanochat/base_data_climbmix/shard_00002.parquet (already exists)
Skipping /root/.cache/nanochat/base_data_climbmix/shard_00003.parquet (already exists)
Skipping /root/.cache/nanochat/base_data_climbmix/shard_00004.parquet (already exists)
Skipping /root/.cache/nanochat/base_data_climbmix/shard_00005.parquet (already exists)
Skipping /root/.cache/nanochat/base_data_climbmix/shard_00006.parquet (already exists)
Skipping /root/.cache/nanochat/base_data_climbmix/shard_00007.parquet (already exists)
Skipping /root/.cache/nanochat/base_data_climbmix/shard_06542.parquet (already exists)
Done! Downloaded: 9/9 shards to /root/.cache/nanochat/base_data_climbmix
max_chars: 2,000,000,000
doc_cap: 10,000
vocab_size: 32,768
Training time: 237.59s
S

2026-06-04 04:21:12,291 - rustbpe - INFO - Processing sequences from iterator (buffer_size: 8192)
2026-06-04 04:24:52,030 - rustbpe - INFO - Processed 677888 sequences total, 1952346 unique
2026-06-04 04:24:52,217 - rustbpe - INFO - Starting BPE training: 32503 merges to compute
2026-06-04 04:24:52,217 - rustbpe - INFO - Computing initial pair counts from 1952346 unique sequences
2026-06-04 04:24:56,461 - rustbpe - INFO - Building heap with 18999 unique pairs
2026-06-04 04:24:56,463 - rustbpe - INFO - Starting merge loop
2026-06-04 04:25:01,246 - rustbpe - INFO - Progress: 1% (326/32503 merges) - Last merge: (276, 115) -> 581 (frequency: 547375)
2026-06-04 04:25:02,177 - rustbpe - INFO - Progress: 2% (651/32503 merges) - Last merge: (316, 291) -> 906 (frequency: 223745)
2026-06-04 04:25:02,788 - rustbpe - INFO - Progress: 3% (976/32503 merges) - Last merge: (257, 477) -> 1231 (frequency: 139428)
2026-06-04 04:25:03,301 - rustbpe - INFO - Progress: 4% (1301/32503 merges) - Last merge: (

Pretrain the small model

In [ ]:
import os
os.environ["NANOCHAT_BASE_DIR"] = os.environ["HOME"] + "/.cache/nanochat"
os.environ["PATH"] = os.environ["HOME"] + "/.local/bin:" + os.environ["PATH"]

!cd nanochat && source .venv/bin/activate && python -m scripts.base_train \
    --depth=6 \
    --head-dim=64 \
    --window-pattern=L \
    --max-seq-len=512 \
    --device-batch-size=32 \
    --total-batch-size=16384 \
    --eval-every=100 \
    --eval-tokens=524288 \
    --core-metric-every=-1 \
    --sample-every=-1 \
    --num-iterations=800 \
    --run=dummy


                                                       █████                █████
                                                      ░░███                ░░███
     ████████    ██████   ████████    ██████   ██████  ░███████    ██████  ███████
    ░░███░░███  ░░░░░███ ░░███░░███  ███░░███ ███░░███ ░███░░███  ░░░░░███░░░███░
     ░███ ░███   ███████  ░███ ░███ ░███ ░███░███ ░░░  ░███ ░███   ███████  ░███
     ░███ ░███  ███░░███  ░███ ░███ ░███ ░███░███  ███ ░███ ░███  ███░░███  ░███ ███
     ████ █████░░████████ ████ █████░░██████ ░░██████  ████ █████░░███████  ░░█████
    ░░░░ ░░░░░  ░░░░░░░░ ░░░░ ░░░░░  ░░░░░░   ░░░░░░  ░░░░ ░░░░░  ░░░░░░░░   ░░░░░
    
Autodetected device type: cuda
2026-06-04 04:26:38,130 - nanochat.common - INFO - Distributed world size: 1
2026-06-04 04:26:38,131 - nanochat.common - WARNING - Peak flops undefined for: Tesla T4, MFU will show as 0%
GPU: Tesla T4 | Peak FLOPS (BF16): inf
COMPUTE_DTYPE: torch.float32 (auto-detected: CUDA SM 75 (pre-Ampere, bf16 no

Supervised fine tunining

In [ ]:
%%bash
export NANOCHAT_BASE_DIR="$HOME/.cache/nanochat"
cd nanochat
source .venv/bin/activate

curl -L -o $NANOCHAT_BASE_DIR/identity_conversations.jsonl \
    https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl

TORCH_DTYPE=float32 python -m scripts.chat_sft \
    --max-seq-len=256 \
    --device-batch-size=16 \
    --total-batch-size=16384 \
    --eval-every=200 \
    --eval-tokens=524288 \
    --num-iterations=800 \
    --run=dummy

Autodetected device type: cuda
COMPUTE_DTYPE: torch.float32 (auto-detected: CUDA SM 75 (pre-Ampere, bf16 not supported, using fp32))
GPU: Tesla T4 | Peak FLOPS (BF16): inf
NOTE: --max-seq-len=256 overrides pretrained value of 512
NOTE: --device-batch-size=16 overrides pretrained value of 32
Using total_batch_size=16384
Inherited embedding_lr=0.3 from pretrained checkpoint
Inherited unembedding_lr=0.008 from pretrained checkpoint
Inherited matrix_lr=0.02 from pretrained checkpoint
Tokens / micro-batch / rank: 16 x 256 = 4,096
Tokens / micro-batch: 4,096
Total batch size 16,384 => gradient accumulation steps: 4
Scaling the LR for the AdamW parameters ∝1/√(384/768) = 1.414214
Loaded optimizer state from pretrained checkpoint (momentum buffers only, LRs reset)
Training mixture: 1,071,759 rows (MMLU x3, GSM8K x4)
Step 00000 | Validation bpb: 1.2999
step 00001 (0.62%) | loss: nan | lrm: 1.00 | dt: 7350.12ms | tok/sec: 2,229 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00002 (1.12%) | loss

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2452k  100 2452k    0     0  1716k      0  0:00:01  0:00:01 --:--:-- 1717k
2026-06-04 05:38:12,581 - nanochat.common - INFO - Distributed world size: 1
2026-06-04 05:38:12,582 - nanochat.common - WARNING - Peak flops undefined for: Tesla T4, MFU will show as 0%
2026-06-04 05:38:12,583 - nanochat.checkpoint_manager - INFO - No model tag provided, guessing model tag: d6
2026-06-04 05:38:12,583 - nanochat.checkpoint_manager - INFO - Loading model from /root/.cache/nanochat/base_checkpoints/d6 with step 800
2026-06-04 05:38:13,208 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 512, 'vocab_size': 32768, 'n_layer': 6, 'n_head': 6, 'n_kv_head': 6, 'n_embd': 384, 'window_pattern': 'L'}
2026-06-04 05:38:16,358 - nanochat.checkpoint_manager - INFO - Loading optimizer state from /root/.cache/nanochat/

CalledProcessError: Command 'b'export NANOCHAT_BASE_DIR="$HOME/.cache/nanochat"\ncd nanochat\nsource .venv/bin/activate\n\ncurl -L -o $NANOCHAT_BASE_DIR/identity_conversations.jsonl \\\n    https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl\n\nTORCH_DTYPE=float32 python -m scripts.chat_sft \\\n    --max-seq-len=256 \\\n    --device-batch-size=16 \\\n    --total-batch-size=16384 \\\n    --eval-every=200 \\\n    --eval-tokens=524288 \\\n    --num-iterations=800 \\\n    --run=dummy\n'' returned non-zero exit status 1.

Chat with model

In [ ]:
%%bash
export NANOCHAT_BASE_DIR="$HOME/.cache/nanochat"
cd nanochat
source .venv/bin/activate

python -m scripts.chat_cli -p "What is the capital of France?"

Autodetected device type: cuda


2026-06-04 05:16:05,498 - nanochat.common - INFO - Distributed world size: 1
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/drive/MyDrive/Demo/CS4182_NLP/nanochat/scripts/chat_cli.py", line 27, in <module>
    model, tokenizer, meta = load_model(args.source, device, phase="eval", model_tag=args.model_tag, step=args.step)
  File "/content/drive/MyDrive/Demo/CS4182_NLP/nanochat/nanochat/checkpoint_manager.py", line 172, in load_model
    return load_model_from_dir(checkpoints_dir, *args, **kwargs)
  File "/content/drive/MyDrive/Demo/CS4182_NLP/nanochat/nanochat/checkpoint_manager.py", line 152, in load_model_from_dir
    model_tag = find_largest_model(checkpoints_dir)
  File "/content/drive/MyDrive/Demo/CS4182_NLP/nanochat/nanochat/checkpoint_manager.py", line 120, in find_large

CalledProcessError: Command 'b'export NANOCHAT_BASE_DIR="$HOME/.cache/nanochat"\ncd nanochat\nsource .venv/bin/activate\n\npython -m scripts.chat_cli -p "What is the capital of France?"\n'' returned non-zero exit status 1.

In [ ]:
import shutil
shutil.copytree(
    '/root/.cache/nanochat/base_checkpoints/d6',
    '/content/drive/MyDrive/nanochat_checkpoints/d6'
)

'/content/drive/MyDrive/nanochat_checkpoints/d6'